# Report (draft)
Feel free to edit this!

## Part A - File Generation

In [ ]:
import os

sizes = [8,64,512,4096,32768,262144,2097152]

for s in sizes:
    with open(f"file_{s}.bin","wb") as f:
        f.write(os.urandom(s))

This code was used to generate binary files of the requested byte sizes. Binary files were used so the input consists of uniformly random byte sequences, avoiding any encoding-related bias that could arise from text-based files.

## Part D - SHA-256 Measurements

In [ ]:
import hashlib
import timeit
import statistics

def sha256_file(filepath, runs=100):
    with open(filepath, "rb") as f:
        data = f.read()

    timer = timeit.Timer(lambda: hashlib.sha256(data).digest())

    # Warm-up
    timer.timeit(number=50)

    raw_times = timer.repeat(repeat=runs, number=1)

    return [t * 1000000 for t in raw_times]

The function above reads the entire file into memory and measures the execution time of SHA-256 hashing. A warm-up phase of 50 runs is performed before any real measurements to reduce the impact of the execution overhead from the Python runtime and the CPU caching effects. The function returns a list of how long each individual hash took, in microseconds.

In [ ]:
import sha_crypto
import statistics
import matplotlib.pyplot as plt
import pandas as pd
import math

results = []

print("File Size       | Average Time | Median Time  | Std Dev    | CI (95%)")
print("----------------+--------------+--------------+------------+---------")

files = [
    (8, "file_8.bin"),
    (64, "file_64.bin"),
    (512, "file_512.bin"),
    (4096, "file_4096.bin"),
    (32768, "file_32768.bin"),
    (262144, "file_262144.bin"),
    (2097152, "file_2097152.bin")
]

for size, f in files:
    
    if size <= 512:
        runs = 1000
    else:
        runs = 500
    
    all_times = sha_crypto.sha256_file(f, runs=runs)
    
    avg_time = statistics.mean(all_times)
    med_time = statistics.median(all_times)
    std_dev = statistics.stdev(all_times)
    
    n = len(all_times)
    ci_95 = 1.96 * (std_dev / math.sqrt(n))
    
    print(f"{size:<15} | {avg_time:<12.2f} | {med_time:<12.2f} | {std_dev:<10.2f} | ±{ci_95:.2f}")
    
    results.append({
        "Size": size, 
        "Mean": avg_time, 
        "StdDev": std_dev,
        "CI": ci_95
    })


This function evaluates SHA-256 performance across different file sizes by repeatedly hashing each file using `sha_crypto.sha256_file()` and recording the execution times for each run.

For file sizes above 512 bytes, 500 independent runs are performed. For smaller file sizes (8, 64, and 512 bytes), 1000 runs are used to improve statistical significance This is necessary because at small input sizes, the SHA-256 computation time is very low and becomes comparable to system overhead (function calls, OS scheduling, and timing resolution). Increasing the number of samples reduces the impact of this noise by decreasing the standard error of the mean.

For each file size, the following statistical metrics are computed from the collected samples:

- **Mean:** the true average execution time for SHA-256 at a given input size used for comparing performance across file sizes.
- **Median:** reduces the influence of occasional slow or fast outliers caused by system noise such as scheduling delays or cache effects.
- **Standard deviation:** reflects variability introduced by system-level noise and runtime fluctuations across repeated executions.
- ***95%* confidence interval:** an estimate of the precision of the mean, indicating the range in which the true average execution time is expected to lie. This is used to assess whether observed differences between file sizes are statistically meaningful.

These metrics allow the results to be interpreted in terms of statistical reliability rather than single-run measurements, supporting meaningful comparison across file sizes.

Finally, the processed results are stored and passed to a plotting stage (implemented using matplotlib and pandas via Gemini) to generate visualisations of SHA-256 execution time scaling with input size.